# Time-series forecasting with SAP-RPT-TS
## Single time-series
This notebook belogs to the <a href='https://community.sap.com/t5/technology-blog-posts-by-sap/hands-on-tutorial-improve-your-time-series-forecasts-with-sap-s-tabular-ai/ba-p/14514276'>Hands-on Tutorial: Improve your time-series forecasts with SAP's Tabular AI model SAP-RPT-TS</a>

### SAP AI Core details

Service key for SAP AI Core and url of deployed model

In [ ]:
# Service Key for SAP AI Core, downloaded from the BTP Cockpit
aicore_servicekey_file_path = 'aicore-servicekey.json'

# Deployment url of the deployed model on SAP AI Core
# SAP AI Launchpad -> ML Operations -> Deployments -> Click the model and you see the URL
DEPLOYMENT_URL_SAPRPTTS = "YOURURLHERE"

In [2]:
import json
from pathlib import Path
aicore_servicekey = Path(aicore_servicekey_file_path)
with aicore_servicekey.open() as f:
    sk = json.load(f)

AICORE_AUTH_URL      = sk["url"]
AICORE_CLIENT_ID     = sk["clientid"]
AICORE_CLIENT_SECRET = sk["clientsecret"]

Obtain Bearer token to access SAP AI Core API

In [3]:
import os
import requests
response = requests.post(
    f"{AICORE_AUTH_URL}/oauth/token",
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    data={
        "grant_type": "client_credentials",
        "client_id": AICORE_CLIENT_ID,
        "client_secret": AICORE_CLIENT_SECRET,
    },
)
response.raise_for_status()
AUTH_TOKEN = response.json()["access_token"]

### Access and prepare data
Load data from CSV file

In [4]:
import pandas as pd
data_df = pd.read_csv('HOTELNIGHTS.csv')
data_df.head(5)

,MONTH,HOTELNIGHTS
0,2020-01-01,3037206
1,2020-02-01,3332477
2,2020-03-01,1271032
3,2020-04-01,218183
4,2020-05-01,643629


In [5]:
data_df.shape

(75, 2)

Sort by month

In [6]:
data_df = data_df.sort_values(by='MONTH')

Plot time-series

In [7]:
import plotly.express as px
fig = px.line(data_df, x="MONTH", y="HOTELNIGHTS", title='Hotel nights Switzerland')
fig.show()

Drop data before 2022 to eliminate Covid time

In [8]:
data_df = data_df[data_df['MONTH'] >= '2022-01-01']
data_df.head(5)

,MONTH,HOTELNIGHTS
24,2022-01-01,2204984
25,2022-02-01,2892697
26,2022-03-01,3053960
27,2022-04-01,2523861
28,2022-05-01,2820085


In [9]:
data_df.shape

(51, 2)

Keep the last 12 months as holdout. Predict the known past to assess forecast accuracy

In [10]:
months_to_forecast=12
train_df = data_df.head(data_df.shape[0] - months_to_forecast)
holdout_df = data_df.tail(months_to_forecast)

### Get forecast from SAP-RPT-TS on SAP AI Core

Call RPT-TS to obtain the prediction

In [ ]:
import requests
import os

# Payload data
payload = {
    "horizon": months_to_forecast,
    "data": [{
        "timestamp": train_df['MONTH'].tolist(),
        "target":    [train_df['HOTELNIGHTS'].tolist()]
    }],
    "options": {"confidence_interval": 0.95,
                "enhanced_forecast": True},
}

# Headers
headers = {
    'AI-Resource-Group': 'default',
    'Authorization': f'Bearer {AUTH_TOKEN}',
    'Content-Type': 'application/json'
}

# Make the POST request
response = requests.post(
    f"{DEPLOYMENT_URL_SAPRPTTS}/predict",
    headers=headers,
    json=payload
)

# Print the response
print(response.status_code)

200


In [12]:
response.json()

{'all_null_targets': [],
 'confidence_interval_output': [[[[2367523.0,
     2813025.0,
     3454048.75,
     4016414.5,
     3907504.0,
     3442129.25,
     2811185.25,
     1676956.375,
     2735209.5,
     2544284.25,
     2931105.75,
     2857055.75],
    [3515779.75,
     4017263.75,
     4891629.5,
     5956058.5,
     5851060.0,
     4985346.5,
     4260247.5,
     3262985.0,
     4108214.75,
     3916972.5,
     4384422.5,
     4293469.0]]]],
 'detected_interval': [{'unit': 'month', 'value': 1}],
 'group_results': None,
 'hierarchical_result': None,
 'quantile_result': None,
 'result': [[[2992700.5,
    3413191.25,
    4144536.5,
    4942853.5,
    4831929.0,
    4191962.5,
    3491679.75,
    2559884.5,
    3383609.5,
    3251664.5,
    3565796.25,
    3530905.5]]],
 'result_timestamp': [['2025-04-01',
   '2025-05-01',
   '2025-06-01',
   '2025-07-01',
   '2025-08-01',
   '2025-09-01',
   '2025-10-01',
   '2025-11-01',
   '2025-12-01',
   '2026-01-01',
   '2026-02-01',
   '202

### Format and analyse the prediction data

Collect the prediction and the lower/upper bounds in a Pandas DataFrame

In [13]:
predicted_df = pd.DataFrame({'MONTH': response.json()['result_timestamp'][0]})
predicted_df.insert(1, "PREDICTION_LOWERBOUND", response.json()['confidence_interval_output'][0][0][0])
predicted_df.insert(2, "PREDICTION", response.json()['result'][0][0])
predicted_df.insert(3, "PREDICTION_UPPERBOUND", response.json()['confidence_interval_output'][0][0][1])
predicted_df

,MONTH,PREDICTION_LOWERBOUND,PREDICTION,PREDICTION_UPPERBOUND
0,2025-04-01,2367523.000,2992700.50,3515779.75
1,2025-05-01,2813025.000,3413191.25,4017263.75
2,2025-06-01,3454048.750,4144536.50,4891629.50
3,2025-07-01,4016414.500,4942853.50,5956058.50
4,2025-08-01,3907504.000,4831929.00,5851060.00
5,2025-09-01,3442129.250,4191962.50,4985346.50
6,2025-10-01,2811185.250,3491679.75,4260247.50
7,2025-11-01,1676956.375,2559884.50,3262985.00
8,2025-12-01,2735209.500,3383609.50,4108214.75
9,2026-01-01,2544284.250,3251664.50,3916972.50


Calculate the MAPE

In [14]:
from sklearn.metrics import mean_absolute_percentage_error
y_true = holdout_df['HOTELNIGHTS']
y_pred = predicted_df[['PREDICTION']]
mean_absolute_percentage_error(y_true, y_pred)


0.03192210960238374

Plot predictions against actuals

In [15]:
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=pd.concat([predicted_df['MONTH'], predicted_df['MONTH'][::-1]]),
    y=pd.concat([predicted_df['PREDICTION_UPPERBOUND'], predicted_df['PREDICTION_LOWERBOUND'][::-1]]),
    fill='toself', fillcolor='rgba(0,200,0,0.2)', line=dict(color='rgba(255,255,255,0)'), name='95% Confidence Interval'))
fig.add_trace(go.Scatter(x=train_df['MONTH'], y=train_df['HOTELNIGHTS'], mode='lines', name='Actuals (train)', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=holdout_df['MONTH'], y=holdout_df['HOTELNIGHTS'], mode='lines', name='Actuals (holdout)', line=dict(color='orange')))
fig.add_trace(go.Scatter(x=predicted_df['MONTH'], y=predicted_df['PREDICTION'], mode='lines', name='Prediction', line=dict(color='green')))
fig.update_layout(title='Hotel Nights: Actuals vs Forecast', xaxis_title='Month', yaxis_title='Hotel Nights')
fig.show()